In [ ]:
# ---- Step 1: Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Step 2: Imports ----
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ---- Step 3: Load data (update this path to match your Drive location) ----
data = pd.read_csv('/content/drive/MyDrive/merged_eeg_datas_50_subjects.csv')

# ---- Step 4: Clean and prepare data ----
data = data.dropna(subset=["is_correct"])

y = data["is_correct"].astype(int)
X = data.drop(columns=[
    "is_correct", "task_id", "student_id", "created_at", "neuro_raw_eeg",
])
X = pd.get_dummies(X, columns=["neuro_dominan_rhythm"])

# ---- Step 5: Split into train and test sets ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Step 6: Scale features ----
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ---- Step 7: Hyperparameter grid ----
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    "kernel": ["rbf", "linear", "poly"]
}

# ---- Step 8: Automated tuning with cross-validation ----
grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

train_acc = accuracy_score(y_train, best_model.predict(X_train))
test_acc = accuracy_score(y_test, best_model.predict(X_test))
val_acc = grid_search.best_score_

# ---- Step 9: PRINT THE RESULTS CLEARLY ----
print("\n" + "="*40)
print("RESULTS")
print("="*40)
print(f"Best Parameters:      {grid_search.best_params_}")
print(f"Validation Accuracy:  {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"Training Accuracy:    {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Test Accuracy:        {test_acc:.4f}  ({test_acc*100:.2f}%)")
print("="*40)

print("\nConfusion Matrix (Test):\n", confusion_matrix(y_test, best_model.predict(X_test)))
print("\nClassification Report (Test):\n", classification_report(y_test, best_model.predict(X_test)))

Mounted at /content/drive
Fitting 5 folds for each of 60 candidates, totalling 300 fits


In [ ]:
fv dcxz